In [1]:
import sys
import os
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import numpy as np
import torch
import random
import torch.nn.functional as F

In [2]:
def read_parameters(file_path):
    params = {}

    # First pass: read and store values as strings
    with open(file_path, 'r') as file:
        for line in file:
            # Skip empty lines and comments (if any)
            line = line.strip()
            if line and not line.startswith('#'):
                # Split the line at the equals sign
                key, value = line.split('=')
                key = key.strip()
                value = value.strip()

                # Store the parameter as a string for now
                params[key] = value

    # Second pass: evaluate expressions and convert values
    for key, value in params.items():
        try:
            # Try to evaluate the value as a Python expression using previously defined parameters
            params[key] = eval(value, {}, params)
        except:
            # If it's not an expression, check if it can be converted to float or int
            if '.' in value:
                params[key] = float(value)
            else:
                params[key] = int(value)
        
        # Set the variable in the global namespace with the evaluated value
        globals()[key] = params[key]

    return params

In [3]:
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from ants_move import *
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../../..")))

pygame 2.6.1 (SDL 2.28.4, Python 3.13.3)
Hello from the pygame community. https://www.pygame.org/contribute.html


/home/alepitt/Explorables/.venv/lib/python3.13/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [4]:
work_folder = 'edge_spen'
saving_folder = 'l7'
os.chdir(work_folder)

In [5]:
file_path = saving_folder + '/parameters.txt'  # Path to the parameters file
parameters = read_parameters(file_path)

for key, value in parameters.items():
    print(f"{key} = {value}")

n = 300
Lx = 32
Ly = 32
delta = 0.3
beta = 4.5
eta = 0.01
decay_rate = 0.015
eta_special = 1
n_special = 30
t_transient = 200
tmax = 1200
batch_size = 60
gamma = 0.99
state_dim = (4, 9, 9)
action_dim = 8
action_bound = 1
vision = 9
actor_lr = 5e-06
critic_lr = 5e-05
weight_decay = 0.0
clip_epsilon = 0.2
entropy_coeff = 0.05
policy_steps = 1000
learning_steps = 1
clip_grad = False
use_gae = True
lam = 0.95


In [6]:
env = Ants(n, Lx, Ly, delta, beta, decay_rate, eta, eta_special,
            tmax, n_special, t_transient, vision=state_dim[-1])

In [7]:
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../../..")))
from ppo_ants import *
config_net_file = f"{saving_folder}/networks.yaml"
ppo_ants = PPO_ants( env,
                 gamma, action_bound, actor_lr, critic_lr, batch_size, weight_decay, config_net_file,
                 clip_epsilon, entropy_coeff, policy_steps, learning_steps,
                 state_dim, action_dim, clip_grad, use_gae=True, lam=0.95)
        

In [8]:
load_from_run = True
ppo_ants.setup_networks()
best_weigths = False

if load_from_run and best_weigths:
    ppo_ants.load_weights(saving_folder + '/weights/best_weights')
else:
    if load_from_run:
        ppo_ants.load_weights(saving_folder + '/weights')
    else:
        ppo_ants.critic.load_state_dict(torch.load(f"{saving_folder}/critic_weights.pth", weights_only=True))
        ppo_ants.actor.load_state_dict(torch.load(f"{saving_folder}/actor_weights.pth", weights_only=True))

In [9]:
seed = int(saving_folder[-1])

In [11]:
import csv
import os
import numpy as np
import torch
import random

n_special_list = [1]
n_seeds = 10
beta = 4.5
delta = 0.3
eta_special = 0.0

for n_special in n_special_list:

    for seed in range(n_seeds):

        saving_folder = 'l' + str(seed)

        env = Ants(n, Lx, Ly, delta, beta, decay_rate, eta, eta_special,
                    tmax, n_special, t_transient, vision=state_dim[-1])
                
        ppo_ants = PPO_ants( env,
                        gamma, action_bound, actor_lr, critic_lr, batch_size, weight_decay, config_net_file,
                        clip_epsilon, entropy_coeff, policy_steps, learning_steps,
                        state_dim, action_dim, clip_grad, use_gae=True, lam=0.95)

        ppo_ants.setup_networks()
        ppo_ants.load_weights(saving_folder + '/weights')

        n_bm = 30
        filename = "abm_ants_alone.csv"

        # Intestazioni delle colonne
        header = [
            "seed", "sim_seed", "n_special",
            "pop_mean",
            "pod_mean",
            "rew_mean",
            "lastr_mean"
        ]

        # Se il file non esiste, scrivi l'header
        if not os.path.exists(filename):
            with open(filename, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(header)

        for sim_seed in range(n_bm):
            np.random.seed(sim_seed)
            torch.manual_seed(sim_seed)
            random.seed(sim_seed)

            ppo_ants.env.reset_objects()
            ppo_ants.env.transient()
            trans_pos = env.positions.copy()
            trans_pher = env.pheromones.copy()

            results = ppo_ants.test_episode(max_prob=False, save_pos=True)
            pop = ppo_ants.env.order_parameter_s()
            pod = ppo_ants.env.order_parameter_p()
            rew = np.mean(results[0])
            lastr = np.mean(np.array(results[0])[-50:])

            # Scriviamo direttamente nel CSV
            row_data = [
                seed, sim_seed, n_special,
                pop,  # std non calcolato singolo episodio
                pod,
                rew,
                lastr
            ]

            with open(filename, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(row_data)

        print(f"Risultati salvati per seed {seed}")


Risultati salvati per seed 0
Risultati salvati per seed 1
Risultati salvati per seed 2
Risultati salvati per seed 3
Risultati salvati per seed 4
Risultati salvati per seed 5
Risultati salvati per seed 6
Risultati salvati per seed 7
Risultati salvati per seed 8
Risultati salvati per seed 9


In [ ]:
# load the dataset
import pandas as pd
load = True
if load:
    dfbm = pd.read_csv("benchmark_results_bd.csv")

In [13]:
dfbm_n30 = dfbm[(dfbm['n_special'] == 30)]

In [13]:
res = dfbm.groupby('seed').agg(['mean', 'std']).drop(columns=['sim_seed', 'n_special'])

In [14]:
res.style.format("{:.2f}")

In [15]:
# average over n_special with standard deviation and skip the seeds column
df=dfbm.groupby(['seed']).agg(['mean']).drop(columns=['sim_seed', 'n_special'])

In [16]:
# display the dataframe with 0.01 precision
df.style.format("{:.2f}")

,pop_mean,pod_mean,rew_mean,lastr_mean
,mean,mean,mean,mean
seed,,,,
0,0.19,0.20,-1.87,-1.83
1,0.19,0.20,-1.86,-1.86
2,0.19,0.20,-1.87,-1.86
3,0.19,0.20,-1.86,-1.85
4,0.19,0.20,-1.86,-1.84
5,0.19,0.20,-1.87,-1.88
6,0.19,0.20,-1.88,-1.88
7,0.19,0.20,-1.87,-1.86


In [17]:
df.agg(['mean', 'std'])

,pop_mean,pod_mean,rew_mean,lastr_mean
,mean,mean,mean,mean
mean,0.188488,0.202889,-1.865152,-1.861437
std,0.000000,0.000000,0.006420,0.015768
